# Tabular PyTorch: NNs for Structured Data

Reach for this when you need: 
- Reference for processing tabular data where relationships are non-linear and high-dimensional.
- To implement Entity Embeddings for categorical features.
- Implementation of mixed-type inputs (Categorical + Numerical) in a single model.

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Entity Embeddings for Categories

Instead of one-hot encoding, use a learned vector for each category value. This allows the model to learn relationships (e.g., Sunday and Saturday are similar).

| Feature | Encoding | Usage |
| :--- | :--- | :--- |
| **One-Hot** | Static vectors | High memory, independent features |
| **Entity Emb.** | Learned vectors | Low memory, preserves similarity |

In [ ]:
class TabularModel(nn.Module):
    def __init__(self, emb_sizes, n_cont, out_sz, layers_sz):
        super().__init__()
        # Categorical Embeddings: List of (vocab_size, embedding_dim)
        self.embeds = nn.ModuleList([nn.Embedding(ni, nf) for ni, nf in emb_sizes])
        
        # Numerical Processor (Sequential blocks or simple Linear)
        n_emb = sum(e.embedding_dim for e in self.embeds)
        self.bn_cont = nn.BatchNorm1d(n_cont)
        
        self.layers = nn.Sequential(
            nn.Linear(n_emb + n_cont, layers_sz[0]),
            nn.ReLU(),
            nn.Linear(layers_sz[0], out_sz)
        )

    def forward(self, x_cat, x_cont):
        embeddings = [e(x_cat[:, i]) for i, e in enumerate(self.embeds)]
        x = torch.cat(embeddings, 1)
        x_cont = self.bn_cont(x_cont)
        x = torch.cat([x, x_cont], 1)
        return self.layers(x)

## 2. Choosing Embedding Sizes

Rule of Thumb: embedding space should be `min(50, (vocab_size + 1) // 2)`.

✅ **Use when**: A category has high cardinality (e.g. UserID, StoreID).
❌ **Don't use when**: Cardinality is extremely low (e.g. Gender, Yes/No); one-hot is simpler.

In [ ]:
cat_counts = [100, 50, 10]
emb_sizes = [(c, min(50, (c + 1) // 2)) for c in cat_counts]

### Common Pitfalls
- **Dtypes**: Categorical indices MUST be `torch.long`. Numerical values MUST be `torch.float`.
- **Normalization**: Unlike GBDTs, Neural Nets REQUIRE numerical normalization. ALWAYS apply `StandardScaler` to numerical inputs before feeding them to the model.
- **Dropout**: Tabular data often overfits easily in deep nets; always use `nn.Dropout` generously.

### Key Takeaways
- NNs are powerful for tabular data when specialized in entity embeddings (Fastai approach).
- Combined cat/cont inputs are normalized separately and then concatenated.
- Entity embeddings are often useful as features for other models (e.g. XGBoost input).